In [1]:
import pandas as pd
import mhcgnomes
import uuid
pd.set_option("display.max_columns",500)
PROTEIN_CHECK="^[ACDEFGHIKLMNPQRSTVWY]+$"
HOST_SPECIES = ['human', 'mouse']
CDR3_CHAINS = ['alpha', 'beta']

In [2]:
def fix_mhc_name(allele_str, chain = 'alpha'):
    try:
        # Предварительные проверки и подготовления
        if pd.isna(allele_str):
            return pd.NA     
        allele_first = allele_str.split(" ")[0]
        if allele_first == "B2M":
            return "B2M"
        # Получение конкретной аллели
        parsed_allele = mhcgnomes.parse(allele_first)
        if isinstance(parsed_allele, mhcgnomes.pair.Pair):
            if chain == 'alpha':
                allele = parsed_allele.alpha
            elif chain == 'beta':
                allele = parsed_allele.beta
            else:
                raise ValueError('Unknown chain')
        elif isinstance(parsed_allele, mhcgnomes.allele.Allele):
            allele = parsed_allele
        else:
            raise ValueError('Not allele')
        # Проверка
        if allele.gene.species.name == "Homo sapiens":
            if len(allele.allele_fields) < 2:
                raise ValueError('Too many allele fields. Need at least 2.')
            elif len(allele.allele_fields) == 2:
                return allele.to_string()
            else:
                return allele.restrict_allele_fields(2, drop_annotations=True, drop_mutations=True).to_string()
        elif allele.gene.species.name == "Mus musculus":
            return allele.to_string()
        else:
            raise ValueError('Wrong species. Only human and mouse are allowed.')
    except (mhcgnomes.errors.ParseError, TypeError, ValueError):
        return pd.NA
    except AttributeError:
        print(allele_first)
        return pd.NA

In [3]:
raw_data = pd.read_csv("../../data/raw-data/MIRA/MIRA.csv",sep = ';')
raw_data.head()

,TCR BioIdentity,Experiment,CDR3,V,J,ORF,epitope,Cell Type,allele_type,allele,type
0,2166104e-6c65-484e-bb95-880273c7a515,eAV93,CASSAQGTGDRGYTF,TCRBV27-01,TCRBJ01-02,"ORF1ab,surface glycoprotein",ADAGFIKQY,naive_CD8,I,A*11:01,peptide
1,2166104e-6c65-484e-bb95-880273c7a515,eAV93,CASSAQGTGDRGYTF,TCRBV27-01,TCRBJ01-02,"ORF1ab,surface glycoprotein",ADAGFIKQY,naive_CD8,I,A*68:01,peptide
2,2166104e-6c65-484e-bb95-880273c7a515,eAV93,CASSAQGTGDRGYTF,TCRBV27-01,TCRBJ01-02,"ORF1ab,surface glycoprotein",ADAGFIKQY,naive_CD8,I,B*35:01,peptide
3,2166104e-6c65-484e-bb95-880273c7a515,eAV93,CASSAQGTGDRGYTF,TCRBV27-01,TCRBJ01-02,"ORF1ab,surface glycoprotein",ADAGFIKQY,naive_CD8,I,B*35:03,peptide
4,2166104e-6c65-484e-bb95-880273c7a515,eAV93,CASSAQGTGDRGYTF,TCRBV27-01,TCRBJ01-02,"ORF1ab,surface glycoprotein",ADAGFIKQY,naive_CD8,I,C*03:03,peptide


In [5]:
raw_data.shape

(1697778, 11)

In [4]:
raw_data['allele'].unique()

array(['A*11:01', 'A*68:01', 'B*35:01', 'B*35:03', 'C*03:03', 'C*04:01',
       'A*02:01', 'A*33:03', 'B*53:01', 'B*58:01', 'C*03:02', 'C*06:02',
       'A*01:01:01', 'A*02:01:01', 'B*15:01:01', 'B*51:01:01',
       'C*01:02:01', 'C*02:02:02', 'B*13:02:01', 'B*18:01:01',
       'C*06:02:01', 'C*07:01:01', 'A*03:01:01', 'B*07:02:01',
       'C*07:02:01', 'A*32:01:01', 'C*04:01:01', 'A*03:01', 'B*07:02',
       'B*44:27', 'C*07:02', 'C*07:04', 'A*02:13', 'A*23:01:01',
       'B*44:02:01', 'B*44:03:01', 'C*04:09N', 'C*05:01:01', 'A*11:01:01',
       'C*03:04:01', 'A*26:01:01', 'B*27:05:02', 'A*29:02:01',
       'B*15:02:01', 'C*04:03:01', 'C*16:01:01', 'A*31:01:02',
       'B*35:01:01', 'A*29:01', 'B*07:05', 'B*18:01', 'C*15:05',
       'B*44:27:01', 'C*07:04:01', 'C*14:02:01', 'A*68:01:02',
       'B*08:01:01', 'B*52:01:01', 'C*12:02:02', 'A*26:01', 'B*44:02',
       'B*52:01', 'C*05:01', 'B*57:01:01', 'B*40:01', 'B*44:03',
       'C*03:04', 'C*16:01', 'A*24:02:01', 'B*27:05', 'C*01:02',

In [10]:
import zipfile
import os
import requests
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
mira_url = 'https://adaptivepublic.blob.core.windows.net/publishedproject-supplements/covid-2020/ImmuneCODE-MIRA-Release002.1.zip'
output = "/home/asmirnov/dissertation_new/TCRpred/data/raw-data/MIRA"
file_name = os.path.join(output, f"MIRA_{date}.zip")
response = requests.get(mira_url)
print("Download...")

if response.status_code == 200:
    with open(file_name, "wb") as file:
        file.write(response.content)
        print(f"Downloaded {file_name}")
else:
    raise ValueError(f"Failed to download file: {response.status_code} - {response.text}")
            
with zipfile.ZipFile(file_name, 'r') as zip_ref:
    zip_ref.extractall(output)
            
home = os.path.join(output,"ImmuneCODE-MIRA-Release002.1")
print(home)

Download...
Downloaded /home/asmirnov/dissertation_new/TCRpred/data/raw-data/MIRA/MIRA_2026-03-24.zip
/home/asmirnov/dissertation_new/TCRpred/data/raw-data/MIRA/ImmuneCODE-MIRA-Release002.1


In [11]:
subject_metadata = pd.read_csv(os.path.join(home,"subject-metadata.csv"),encoding = "cp1251")
subject_metadata.head()

,Experiment,Subject,Cell Type,Target Type,Cohort,Age,Gender,Race,HLA-A,HLA-A.1,HLA-B,HLA-B.1,HLA-C,HLA-C.1,DPA1,DPA1.1,DPB1,DPB1.1,DQA1,DQA1.1,DQB1,DQB1.1,DRB1,DRB1.1,DRB3,DRB3.1,DRB4,DRB4.1,DRB5,DRB5.1
0,eAM13,844,PBMC,C19_cI,COVID-19-Convalescent,34.0,F,White,A*02:01:01,A*02:05:01,B*08:01:01,B*41:01:01,C*07:01:01,C*07:01:01,DPA1*01:03:01,DPA1*02:01:02,DPB1*01:01:01,DPB1*02:01:02,DQA1*05:01:01,DQA1*05:05:01,DQB1*02:01:01,DQB1*03:01:01,DRB1*03:01:01,DRB1*13:05:01,DRB3*01:01:02,DRB3*02:02:01,NaN,NaN,NaN,NaN
1,eAM23,5422,PBMC,C19_cI,COVID-19-Convalescent,48.0,M,NaN,A*11:01:01,A*24:02:01,B*15:01:01,B*52:01:01,C*04:01:01,C*12:02:02,DPA1*02:02:02,DPA1*02:02:02,DPB1*05:01:01,DPB1*05:01:01,DQA1*01:03:01,DQA1*03:01:01,DQB1*03:02:01,DQB1*06:01:01,DRB1*04:06:01,DRB1*15:02:01,DRB4*01:03:01,DRB5*01:02,NaN,NaN,NaN,NaN
2,eAV100,1995,PBMC,C19_cII,COVID-19-Convalescent,29.0,F,NaN,A*02:01:01,A*68:01:02,B*07:02:01,B*40:01:02,C*03:04:01,C*07:02:01,DPA1*01:03:01,DPA1*01:03:01,DPB1*04:01:01,DPB1*04:01:01,DQA1*01:02:01,DQA1*05:05:01,DQB1*03:01:01,DQB1*06:02:01,DRB1*11:01:01,DRB1*15:01:01,DRB3*02:02:01,NaN,NaN,NaN,DRB5*01:01:01,NaN
3,eAV105,1995,PBMC,C19_cII,COVID-19-Convalescent,29.0,F,NaN,A*02:01:01,A*68:01:02,B*07:02:01,B*40:01:02,C*03:04:01,C*07:02:01,DPA1*01:03:01,DPA1*01:03:01,DPB1*04:01:01,DPB1*04:01:01,DQA1*01:02:01,DQA1*05:05:01,DQB1*03:01:01,DQB1*06:02:01,DRB1*11:01:01,DRB1*15:01:01,DRB3*02:02:01,NaN,NaN,NaN,DRB5*01:01:01,NaN
4,eAV88,19830,naive_CD8,C19_cI,Healthy (No known exposure),24.0,M,White,A*02:01,A*03:01,B*27:05,B*40:01,C*03:04,C*07:04,DPA1*01:03,DPA1*01:04,DPB1*02:01,DPB1*15:01,DQA1*03:01,DQA1*03:02,DQB1*03:02,DQB1*03:03,DRB1*04:04,DRB1*09:01,NaN,NaN,DRB4*01:03,DRB4*01:03,NaN,NaN


In [12]:
subject_metadata.shape

(144, 30)

In [13]:
subject_metadata = pd.read_csv(os.path.join(home,"peptide-hits-ci.csv"),encoding = "cp1251")
subject_metadata.head()

,ORF,Amino Acids,Start Index in Genome,End Index in Genome,Hits
0,"ORF1ab,surface glycoprotein","ADAGFIKQY,AELEGIQY,LADAGFIKQY,TLADAGFIK",533,24073,112
1,ORF1ab,"GEIPVAYRKVLL,VPHVGEIPVAY",587,634,721
2,ORF1ab,SEVGPEHSLAEY,1391,1426,325
3,ORF1ab,"AIILASFSA,ILASFSAST",1685,1717,15
4,ORF1ab,TSDLATNNLVVMAY,2024,2065,74
